# Proyek Analisis Data: E-Commerce

- **Nama:** ISI NAMA ANDA  
- **Email:** ISI EMAIL ANDA  
- **ID Dicoding:** ISI ID DICODING ANDA

Notebook ini menyajikan analisis mandiri atas dataset Brazilian E-Commerce (Olist). Fokus analisis adalah pengalaman pengiriman dan segmentasi pelanggan untuk retensi. Seluruh angka dihitung ulang dari CSV lokal.

## Menentukan Pertanyaan Bisnis

Periode pembelian dibatasi pada **1 Januari 2017–31 Agustus 2018**. Batas waktu ini membuat perbandingan konsisten dan tanggal acuan RFM ditetapkan pada 1 September 2018.

1. **Pengiriman & ulasan:** Pada periode tersebut, state pelanggan mana yang mempunyai tingkat keterlambatan tertinggi dengan minimal 100 pesanan yang dapat dievaluasi, dan bagaimana proporsi ulasan buruk berbeda antara pesanan terlambat dan tepat waktu?
2. **RFM:** Dengan acuan 1 September 2018, bagaimana pelanggan delivered dikelompokkan berdasarkan recency, frequency, dan monetary untuk menentukan sasaran retensi atau aktivasi ulang?

Kedua pertanyaan spesifik, terukur, dapat ditindaklanjuti, relevan terhadap operasi e-commerce, dan memiliki batas waktu yang jelas.

## Import Library dan Konfigurasi

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'analysis.py').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis import (
    ANALYSIS_START, ANALYSIS_END, REFERENCE_DATE, MIN_STATE_ORDERS,
    assess_data, build_rfm, clean_and_merge, delivery_by_state,
    load_raw_data, review_by_delivery_status, rfm_summary
)
plt.style.use('seaborn-v0_8-whitegrid')

## Data Wrangling

### Gathering Data
Empat tabel digunakan: orders sebagai basis pesanan, customers untuk identitas pelanggan dan state, order items untuk nilai barang, serta reviews untuk kepuasan pelanggan.

In [ ]:
raw = load_raw_data()
for name, frame in raw.items():
    print(f'{name}: {frame.shape[0]:,} baris x {frame.shape[1]} kolom')
    display(frame.head(3))

### Assessing Data
Pemeriksaan berikut mencari setidaknya dua persoalan kualitas: nilai kosong dan duplikasi persis. Selain itu, tanggal dan nilai harga akan diparsing pada tahap cleaning. Review dapat muncul lebih dari sekali dalam satu order; tanpa aturan pemilihan, join review dapat menggandakan order.

In [ ]:
quality = assess_data(raw)
display(quality)
print('Order tanpa tanggal penerimaan:', raw['orders']['order_delivered_customer_date'].isna().sum())
print('Produk tanpa kategori:', 'Tidak diperiksa karena kategori tidak dipakai oleh dua pertanyaan ini')
print('Order dengan lebih dari satu review:', (raw['reviews'].groupby('order_id').size() > 1).sum())

### Cleaning Data
Tanggal diubah menjadi datetime dengan `errors='coerce'`; nilai tidak valid menjadi kosong, bukan ditebak. Hanya satu review terbaru per order yang dipilih agar tabel hasil tetap ber-grain satu order. Tanggal penerimaan atau estimasi yang kosong tidak diimputasi dan order tersebut hanya dikeluarkan dari metrik keterlambatan, bukan dihapus dari seluruh analisis. Harga item dikonversi numerik sebelum dijumlahkan menjadi monetary.

In [ ]:
orders, items = clean_and_merge(raw)
display(orders[['order_id', 'customer_state', 'order_purchase_timestamp', 'is_late', 'review_score', 'item_value_brl']].head())
print('Order dalam periode:', orders['is_in_period'].sum())
print('Order delivered dengan tanggal pengiriman lengkap:', (orders['is_in_period'] & orders['is_delivered'] & orders['delivery_is_known']).sum())

## Exploratory Data Analysis

### Pertanyaan 1 — Pengiriman dan ulasan
Keterlambatan didefinisikan ketika tanggal kalender penerimaan melewati tanggal estimasi. Ulasan buruk adalah skor 1–2. Peringkat state menggunakan minimum 100 order agar state dengan sampel sangat kecil tidak menjadi prioritas hanya karena variasi acak.

In [ ]:
state_delivery = delivery_by_state(orders)
review_delivery = review_by_delivery_status(orders)
priority_states = state_delivery.query('eligible_for_priority').head(10)
display(priority_states)
display(review_delivery)

## Visualization & Explanatory Analysis

Grafik pertama menunjukkan state yang layak dibandingkan dan diurutkan menurut tingkat keterlambatan. Grafik kedua menjawab apakah pengalaman pelanggan berbeda pada kelompok pengiriman tepat waktu dan terlambat. Sumbu persentase dimulai dari nol agar perbedaan tidak dibesar-besarkan.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_state = priority_states.sort_values('late_rate_pct')
axes[0].barh(plot_state['customer_state'], plot_state['late_rate_pct'], color='#2F6690')
axes[0].set(title='10 state prioritas menurut keterlambatan', xlabel='Tingkat keterlambatan (%)', ylabel='State')
axes[0].set_xlim(left=0)
axes[1].bar(review_delivery['delivery_status'], review_delivery['bad_review_rate_pct'], color=['#5B8E7D', '#C96C50'])
axes[1].set(title='Proporsi ulasan buruk menurut ketepatan pengiriman', ylabel='Ulasan buruk (%)', xlabel='Status pengiriman')
axes[1].set_ylim(bottom=0)
plt.tight_layout()
plt.show()

### Pertanyaan 2 — Analisis Lanjutan RFM
RFM dihitung pada customer unik dengan order delivered. Recency adalah hari sejak pembelian terakhir hingga 1 September 2018; frequency adalah jumlah order unik; monetary adalah total harga item tanpa ongkir. Segmentasi adalah pengelompokan manual: Champions (baru dan berulang), Pelanggan baru, Pelanggan potensial, Perlu perhatian, dan Tidak aktif. Ini bukan model machine learning atau prediksi churn.

In [ ]:
rfm = build_rfm(orders)
segment_summary = rfm_summary(rfm)
display(segment_summary)
print('Tanggal acuan RFM:', REFERENCE_DATE.date())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(segment_summary['segment'].astype(str), segment_summary['customers'], color='#5B8E7D')
axes[0].set(title='Jumlah pelanggan per segmen RFM', ylabel='Pelanggan')
axes[0].tick_params(axis='x', rotation=25)
axes[1].bar(segment_summary['segment'].astype(str), segment_summary['monetary'], color='#C96C50')
axes[1].set(title='Nilai barang per segmen RFM', ylabel='Monetary (BRL)')
axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

## Conclusion & Recommendation

Jalankan sel berikut setelah seluruh notebook dieksekusi agar kesimpulan menampilkan angka aktual dari dataset. Interpretasi bersifat deskriptif; perbedaan review tidak membuktikan sebab-akibat.

In [ ]:
top_state = priority_states.iloc[0]
late = review_delivery.set_index('delivery_status').loc['Terlambat']
on_time = review_delivery.set_index('delivery_status').loc['Tepat waktu']
largest_segment = segment_summary.sort_values('customers', ascending=False).iloc[0]
print(f"Kesimpulan 1: {top_state['customer_state']} memiliki tingkat keterlambatan tertinggi di antara state dengan minimal {MIN_STATE_ORDERS} order, yaitu {top_state['late_rate_pct']:.2f}%.")
print(f"Kesimpulan 2: Ulasan buruk pada pengiriman terlambat adalah {late['bad_review_rate_pct']:.2f}%, sedangkan pada pengiriman tepat waktu {on_time['bad_review_rate_pct']:.2f}%.")
print(f"Kesimpulan 3: Segmen terbesar adalah {largest_segment['segment']} dengan {int(largest_segment['customers']):,} pelanggan.")
print('Rekomendasi: audit mitra dan proses pengiriman pada state prioritas; kirim kampanye retensi terukur kepada segmen Perlu perhatian/Tidak aktif dan pertahankan Champions dengan program loyalitas.')